In [1]:
import numpy as np
import pandas as pd

from tools.data.store import get_kline
from tools.visual.inspect import analysis, plot_df

In [2]:
df = get_kline('DOGEUSD_PERP', 60)
df['timestamp'] = df.index
for i, n in enumerate([7, 25, 99]):
    df[f'M{i+1}'] = df.close.rolling(n).mean()
df['M4'] = df['M1'].diff()
df['M5'] = (df['M4'] > 0).rolling(5).sum()
df['M4_1'] = df['M4'].shift(1)
df['buy'] = df['sell'] = False

In [3]:
df

,start_t,open,high,low,close,volume,amount,trade_cnt,taker_vol,taker_amt,end_t,timestamp,M1,M2,M3,M4,M5,M4_1,buy,sell
start_t,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,1704067200000,0.08959,0.09010,0.08956,0.08994,55174.0,6.137663e+06,1130,25982.0,2.890516e+06,1704070799999,2024-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2024-01-01 01:00:00,1704070800000,0.08992,0.09035,0.08964,0.09026,82339.0,9.146120e+06,1139,42171.0,4.683946e+06,1704074399999,2024-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2024-01-01 02:00:00,1704074400000,0.09026,0.09030,0.08985,0.08991,66552.0,7.383343e+06,883,21333.0,2.367232e+06,1704077999999,2024-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2024-01-01 03:00:00,1704078000000,0.08990,0.08999,0.08910,0.08933,61512.0,6.871363e+06,1199,30903.0,3.452907e+06,1704081599999,2024-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False
2024-01-01 04:00:00,1704081600000,0.08931,0.08950,0.08854,0.08924,138536.0,1.555806e+07,2533,66195.0,7.434925e+06,1704085199999,2024-01-01 04:00:00,NaN,NaN,NaN,NaN,0.0,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-14 19:00:00,1741978800000,0.17110,0.17235,0.17021,0.17084,219770.0,1.282175e+07,3093,112940.0,6.585137e+06,1741982399999,2025-03-14 19:00:00,0.171823,0.168588,0.165059,0.000144,5.0,0.000201,False,False
2025-03-14 20:00:00,1741982400000,0.17086,0.17088,0.16918,0.17048,219177.0,1.290056e+07,3630,88058.0,5.180249e+06,1741985999999,2025-03-14 20:00:00,0.171694,0.168901,0.165187,-0.000129,4.0,0.000144,False,False
2025-03-14 21:00:00,1741986000000,0.17041,0.17167,0.16997,0.17153,132075.0,7.738467e+06,1950,67512.0,3.953990e+06,1741989599999,2025-03-14 21:00:00,0.171687,0.169275,0.165386,-0.000007,3.0,-0.000129,False,False


In [33]:
import pandas as pd
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

def analysis(df, save_path=None, width=None, height=None):
    fig = go.Figure()

    # Add candlestick
    fig.add_trace(
        go.Candlestick(
            x=df['timestamp'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='DOGE/USD'
        )
    )

    # Add M1 with thicker line
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['M1'],
            line=dict(color='orange', width=1),
            name='M1'
        )
    )

    # Add M2 with thicker line
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['M2'],
            line=dict(color='blue', width=1, dash='dot'),
            name='M2'
        )
    )

    # Add M3 with thicker line
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'],
            y=df['M3'],
            line=dict(color='purple', width=1),
            name='M3'
        )
    )

    # Add Buy/Sell signals
    fig.add_trace(
        go.Scatter(
            x=df[df['buy'] > 0]['timestamp'],
            y=df[df['buy'] > 0]['close'],
            mode='markers',
            marker_symbol='triangle-up',
            marker_color='lime',
            marker_size=11,
            name='Buy Signal'
        )
    )
    fig.add_trace(
        go.Scatter(
            x=df[df['sell'] > 0]['timestamp'],
            y=df[df['sell'] > 0]['close'],
            mode='markers',
        marker_symbol='triangle-down',
        marker_color='blue',
        marker_size=11,
            name='Sell Signal'
        )
    )

    # # Add M4 bar on secondary y-axis
    # fig.add_trace(
    #     go.Bar(
    #         x=df['timestamp'],
    #         y=df['M4'],
    #         name='M4',
    #         yaxis='y2',  # Assign to secondary y-axis
    #         marker_color='gray',
    #         opacity=0.6
    #     )
    # )

    # Configure dual y-axes layout
    fig.update_layout(
        # title='Technical Analysis',
        paper_bgcolor="white",  # 画布背景
        plot_bgcolor="white",   # 绘图区域背景
        margin=dict(l=0, r=0, t=0, b=0),
        xaxis_rangeslider_visible=False,
        xaxis=dict(showticklabels=False),
        yaxis=dict(
            # title='Price (USD)',
            side='left',
            showgrid=True,
            gridcolor='blue',
            showticklabels=False,
        ),
        # yaxis2=dict(
        #     # title='M4 Value',
        #     side='right',
        #     overlaying='y',
        #     showgrid=False,
        #     showticklabels=False,
        # ),
        width=width,
        height=height,
        hovermode='x unified',
        showlegend=False,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    if save_path:
        fig.write_image(save_path)
    return fig

In [34]:
scale = 3
x = analysis(df.tail(168), 'tail.png', 224 * 3, 224 * 3)

In [38]:
x = 104
analysis(df.iloc[x:x+168], 'tail.png', 224, 224)

/miniconda3/lib/python3.10/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



In [42]:
scale = 3
goahead = 1
lookback = 168
returns = df.close.diff()
with open('data/1h/lable.csv', 'w') as fp:
    for i in range(lookback, df.shape[0] - goahead):
        fname = df.index[i].strftime('%Y%m%d%H')
        # analysis(df.iloc[i-lookback:i], f'data/1h/{fname}.png', 224 * scale, 224 * scale)
        fp.write(fname + ',' + ','.join([str(x) for x in returns.iloc[i-24:i+goahead]]) + '\n')
        print(f'{i}/{df.shape[0]}')


168/10536
169/10536
170/10536
171/10536
172/10536
173/10536
174/10536
175/10536
176/10536
177/10536
178/10536
179/10536
180/10536
181/10536
182/10536
183/10536
184/10536
185/10536
186/10536
187/10536
188/10536
189/10536
190/10536
191/10536
192/10536
193/10536
194/10536
195/10536
196/10536
197/10536
198/10536
199/10536
200/10536
201/10536
202/10536
203/10536
204/10536
205/10536
206/10536
207/10536
208/10536
209/10536
210/10536
211/10536
212/10536
213/10536
214/10536
215/10536
216/10536
217/10536
218/10536
219/10536
220/10536
221/10536
222/10536
223/10536
224/10536
225/10536
226/10536
227/10536
228/10536
229/10536
230/10536
231/10536
232/10536
233/10536
234/10536
235/10536
236/10536
237/10536
238/10536
239/10536
240/10536
241/10536
242/10536
243/10536
244/10536
245/10536
246/10536
247/10536
248/10536
249/10536
250/10536
251/10536
252/10536
253/10536
254/10536
255/10536
256/10536
257/10536
258/10536
259/10536
260/10536
261/10536
262/10536
263/10536
264/10536
265/10536
266/10536
267/10536
